# Pruebas del proyecto en Colab con varias bases

Instala PostgreSQL, baja el repo desde GitHub, crea varias bases con el esquema de
`sql/base_de_prueba.sql` (cambiando tamaño y sesgo) y corre las pruebas unitarias,
de integración y E2E sobre ellas. Se corre con **Entorno de ejecución → Ejecutar todas**.

In [ ]:
# Rama del repo y bases a probar. Todas tienen el esquema de sql/base_de_prueba.sql;
# cambia la cantidad de filas y el sesgo (cuántos clientes caen en 5 naciones).
RAMA = "main"
BASES = [
    # nombre,        proveedores, clientes, partes, órdenes, sesgo
    ("optimizacion",  1_000,  15_000, 10_000,  60_000, 0.6),
    ("chica",           100,   1_500,  1_000,   6_000, 0.6),
    ("grande",        4_000,  60_000, 40_000, 240_000, 0.6),
    ("sin_sesgo",     1_000,  15_000, 10_000,  60_000, 0.0),
    ("muy_sesgada",   1_000,  15_000, 10_000,  60_000, 0.95),
]
E2E_EN = ["optimizacion"]   # el navegador demora 2-3 minutos por base

In [ ]:
# Instala PostgreSQL, el repo y sus dependencias (unos 3-4 minutos).
!apt-get -qq update > /dev/null && apt-get -qq install -y postgresql > /dev/null
!apt-get -qq install -y postgresql-$(ls /usr/lib/postgresql | sort -n | tail -1)-hypopg > /dev/null   # para la V3
!service postgresql start
!rm -rf /content/tesis && git clone -q -b {RAMA} https://github.com/FractalLoco/tesis.git /content/tesis
%cd /content/tesis
!pip -q install -r requirements.txt pytest httpx pandas pytest-playwright
!python -m playwright install --with-deps chromium > /dev/null

In [ ]:
# Crea las bases y el usuario de solo lectura que usa la app. La V3 arma su base
# gemela con el usuario postgres (GEMELA_DSN), como lo haría el servidor de la app.
import os
import re
import secrets
import subprocess
import sys

ADMIN = ["sudo", "-u", "postgres", "psql", "-q", "-v", "ON_ERROR_STOP=1"]
USUARIO = "analizador_ro"
CLAVE = secrets.token_urlsafe(16)   # clave de prueba, solo existe en esta sesión
CLAVE_ADMIN = secrets.token_urlsafe(16)

def psql(sql, base="postgres"):
    subprocess.run(ADMIN + ["-d", base], input=sql, text=True, check=True)

def sql_de_la_base(proveedores, clientes, partes, ordenes, sesgo):
    sql = open("sql/base_de_prueba.sql", encoding="utf-8").read()
    cambios = [
        (r"generate_series\(0, 999\) i", f"generate_series(0, {proveedores - 1}) i", 1),
        (r"generate_series\(0, 14999\) i", f"generate_series(0, {clientes - 1}) i", 1),
        (r"generate_series\(0, 9999\) i", f"generate_series(0, {partes - 1}) i", 1),
        (r"generate_series\(0, 59999\)", f"generate_series(0, {ordenes - 1})", 2),
        (r"random\(\) \* 15000\)", f"random() * {clientes})", 1),
        (r"random\(\) \* 10000\)", f"random() * {partes})", 1),
        (r"random\(\) \* 1000\)(?=::int,\s+1 \+)", f"random() * {proveedores})", 1),
        (r"random\(\) < 0\.6", f"random() < {sesgo}", 2),
    ]
    # Primero se marcan los lugares y después se reemplazan, para que un número
    # nuevo no coincida con el patrón de otro cambio.
    for i, (patron, _, veces) in enumerate(cambios):
        sql, n = re.subn(patron, f"@{i}@", sql)
        assert n == veces, f"sql/base_de_prueba.sql cambió, revisar: {patron}"
    for i, (_, nuevo, _) in enumerate(cambios):
        sql = sql.replace(f"@{i}@", nuevo)
    return sql

for nombre, *_ in BASES:
    psql(f"DROP DATABASE IF EXISTS {nombre}")
psql(f"ALTER USER postgres PASSWORD '{CLAVE_ADMIN}'")
psql(f"DROP ROLE IF EXISTS {USUARIO}")
psql(f"CREATE ROLE {USUARIO} LOGIN PASSWORD '{CLAVE}'; GRANT pg_read_all_stats TO {USUARIO};")
for nombre, *tamano in BASES:
    psql(f"CREATE DATABASE {nombre}")
    psql(sql_de_la_base(*tamano), nombre)
    psql(f"GRANT CONNECT ON DATABASE {nombre} TO {USUARIO}; GRANT USAGE ON SCHEMA public TO {USUARIO};"
         f"GRANT SELECT ON ALL TABLES IN SCHEMA public TO {USUARIO};", nombre)
    print("base lista:", nombre)

In [ ]:
# Corre una carpeta de pruebas apuntando la app a una base (igual que el .env).
import pandas as pd

def correr(carpeta, base):
    entorno = {**os.environ, "DB_HOST": "127.0.0.1", "DB_PORT": "5432", "DB_NAME": base,
               "DB_USER": USUARIO, "DB_PASSWORD": CLAVE,
               "GEMELA_DSN": f"host=127.0.0.1 dbname=postgres user=postgres password={CLAVE_ADMIN}"}
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", carpeta],
                       env=entorno, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"--- {carpeta} en {base} ---\n{r.stdout[-3000:]}")
    conteo = {k.rstrip("s"): int(n) for n, k in re.findall(r"(\d+) (passed|failed|skipped|errors?)", r.stdout)}
    return {"base": base, "pruebas": carpeta.split("/")[-1], "pasaron": conteo.get("passed", 0),
            "fallaron": conteo.get("failed", 0) + conteo.get("error", 0), "omitidas": conteo.get("skipped", 0)}

# Las unitarias no usan la base: basta una vez. Integración en todas; E2E en E2E_EN.
resultados = [correr("tests/unitarias", BASES[0][0])]
for nombre, *_ in BASES:
    resultados.append(correr("tests/integracion", nombre))
    if nombre in E2E_EN:
        resultados.append(correr("tests/e2e", nombre))
tabla = pd.DataFrame(resultados)
tabla

In [ ]:
# "omitidas" en integración o E2E significa que no pudo conectarse a la base.
malas = tabla[(tabla.fallaron > 0) | ((tabla.pruebas != "unitarias") & (tabla.omitidas > 0))]
print("Todo pasó en todas las bases." if malas.empty else f"Revisar:\n{malas}")